> **SUPERSEDED (M1 code rev m1.2-r1).** Use `GEMS_M1_CHAMPION_TO_SUBMISSION.ipynb` — the single canonical Milestone-1 notebook. This notebook is kept for history only; its `EXPECT['code_rev']` (m1.1-r1) no longer matches the code, so it stops at the version check by design.

# M1 · 03 — Submission export (CPU)

**Purpose.** Write ONE validated, reproducible GeoTIFF of the champion, built zone by zone from the evaluated held-out maps. No retraining, no new inference, no upload.

| | |
|---|---|
| Runtime | CPU, High-RAM |
| Prerequisites | notebooks 00–02 COMPLETE (artifacts, selection, validation, parity) |
| Inputs | `selection/champion_recipe.yaml`, `validation/validation_status.json` (validated array sha256), the frozen maps, `example_submission.tif` |
| Outputs | `submissions/submission_m1_<family>_<recipe_id>.tif` (+ `.json` sidecar, `_plan.json`, `_validation.json`, `_diagnostics.csv`, `_preview.png`, `_sha256.txt`, `_m1_manifest.json`), canonical `submission_manifest.json`, `submission_validation.json`, `submission_sha256.txt`, `export_parity_report.json`, `README.md` |
| Next | manual upload (only `READY_FOR_UPLOAD`), then `04_LEADERBOARD_REVIEW.ipynb` |

Gates:
* **INCOMPLETE**: an upstream stage is not COMPLETE; nothing is written.
* **BLOCKED**: any check failed; a written file is never labelled ready.
* **READY_FOR_UPLOAD**: every check passed. The checks are:
  * the official checks (`gems3.submission.validate`);
  * the export audit (CRS, transform, shape, NaN pattern, range, eligibility, budget, sidecar);
  * zone coverage and parity;
  * the written band equals the validated array;
  * the written recipe equals the champion recipe;
  * the file size.

## 1 · Configuration

In [ ]:
CONFIG = {'repo': 'mybenkhadda/GEMS---drivendata', 'ref': 'main', 'drive_root': '/content/drive/MyDrive/GEMS_2',
          'local_root': '/content/gems_work', 'reuse_cached': True, 'strict_provenance': True,
          'overwrite_existing': False, 'device': 'cpu'}
PRIVATE_REPO = False
M1_CONFIG = {}
EXPECT = {'code_rev': 'm1.1-r1', 'commit': None}

## 2 · Drive, strict clone, version lock, shared session

In [ ]:
import base64, os, subprocess, sys, json
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
def git(*a, check=True):
    env = None
    if PRIVATE_REPO:
        from google.colab import userdata
        hdr = base64.b64encode(('x-access-token:' + (userdata.get('GITHUB_TOKEN') or '')).encode()).decode()
        env = {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + hdr}
    r = subprocess.run(['git', *a], capture_output=True, text=True, env=env)
    if check and r.returncode:
        raise RuntimeError('git ' + ' '.join(a[:4]) + ' failed:\n' + r.stderr[-1500:])
    return r
if not Path(CODE, '.git').exists():
    git('clone', '-q', 'https://github.com/' + CONFIG['repo'] + '.git', CODE)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin')
if git('-C', CODE, 'rev-parse', '--verify', '-q', 'refs/remotes/origin/' + CONFIG['ref'], check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', '-B', 'm1_colab_active', 'origin/' + CONFIG['ref'])
else:
    git('-C', CODE, 'checkout', '-q', '--detach', CONFIG['ref'])
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules:
    raise RuntimeError('gems_bench already imported: restart the runtime')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', CODE + '[colab]'], check=True)
sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
import gems_bench.leaderboard_focus as LF
assert LF.M1_CODE_REV == EXPECT['code_rev'] and (not EXPECT['commit'] or HEAD.startswith(EXPECT['commit']))
from gems_bench.leaderboard_focus.session import M1Session
m1 = M1Session.attach(CONFIG, M1_CONFIG)
print('HEAD', HEAD, '| statuses:', {s: m1.status(s) for s in ('artifacts', 'selection', 'validation', 'parity')})

## 3 · Export + gate

In [ ]:
from IPython.display import Image, Markdown, display
from gems_bench.leaderboard_focus.submission import export
SUB = export(m1)
print('=' * 100)
print('SUBMISSION STATUS:', SUB.get('status'))
print('FILE (local):', SUB.get('path_local'))
print('FILE (Drive):', CONFIG['drive_root'] + '/' + str(SUB.get('path_drive')))
print('SHA256:', SUB.get('sha256'))
for r in SUB.get('reasons') or []:
    print('  reason:', r)
print('uploaded: False (manual upload only)')
print('=' * 100)
if SUB.get('file'):
    p = m1.ws.path('submissions/' + Path(SUB['file']).stem + '_preview.png')
    if p.exists():
        display(Image(filename=str(p)))
display(Markdown(m1.ws.path('submissions/README.md').read_text(encoding='utf-8')) if m1.ws.path('submissions/README.md').exists() else 'no README (export not run)')

## 4 · Verification checklist

1. `SUBMISSION STATUS: READY_FOR_UPLOAD`. Anything else must not be uploaded.
2. `submission_validation.json`: every M1 check is ok, and the p6 validation status is PASS.
3. `export_parity_report.json`: `equal: true` (the written band is the validated array), `recipe_diff: []`.
4. The Drive copy exists and its sha256 equals `submission_sha256.txt`.

## 5 · Manual upload

1. Download the `.tif` from Drive.
2. Run `sha256sum` and compare the result with `submission_sha256.txt`.
3. On DrivenData, open competition 306 → Submissions and upload the file. The limit is 3 scored submissions per rolling 7 days.
4. Record the public score in notebook 04.

## 6 · Troubleshooting

* `INCOMPLETE`: an earlier notebook did not finish COMPLETE (see `manifests/*.json`).
* `written band != validated array`: an input changed between notebooks 02 and 03. Re-run 02, then 03.
* `FileExistsError`: both `<name>.tif` and `<name>__<fp8>.tif` exist for other recipes. M1 refuses to overwrite; inspect them and archive by hand.
* Template / CRS failure: `example_submission.tif` differs from the feature grid; see `submission/official_spec.json`.

## 7 · Next

After a manual upload: `04_LEADERBOARD_REVIEW.ipynb`.